# Gladiators · Chennai road damage survey → dataset
Runtime → Change runtime type → **T4 GPU**. Run the cells in order.

Input: a ZIP of road photos taken with **location (GPS) ON**. Output: `chennai_damage_survey.csv` (lat, lon, type, size…) for the Gladiators City Road Planner, plus photos with boxes.

## 1. Install and load the road damage model

In [ ]:
!pip install -q ultralytics huggingface_hub pillow-heif folium
from huggingface_hub import hf_hub_download
from ultralytics import YOLO
import pillow_heif; pillow_heif.register_heif_opener()      # iPhone HEIC photos
model = YOLO(hf_hub_download("dronefreak/rdd2022-yolov8s", "best.pt"))
print(model.names)

## 2. Upload the ZIP of survey photos
Send photos to the laptop by **USB cable or Google Drive**, not WhatsApp (WhatsApp removes the GPS).

In [ ]:
from google.colab import files
import zipfile, os, glob, shutil
shutil.rmtree('photos', ignore_errors=True); os.makedirs('photos')
up = files.upload()
for name in up:
    if name.lower().endswith('.zip'):
        zipfile.ZipFile(name).extractall('photos')
    else:
        shutil.move(name, 'photos/' + name)
imgs = sorted(p for p in glob.glob('photos/**/*', recursive=True) if p.lower().endswith(('.jpg', '.jpeg', '.png', '.heic')))
print(len(imgs), 'photos')

## 3. Read GPS from each photo and detect damage

In [ ]:
from PIL import Image
import pandas as pd

def get_gps(path):
    try:
        g = Image.open(path).getexif().get_ifd(0x8825)
    except Exception:
        return None
    if not g or 2 not in g or 4 not in g:
        return None
    dms = lambda v: float(v[0]) + float(v[1]) / 60 + float(v[2]) / 3600
    lat = dms(g[2]) * (-1 if g.get(1) in ('S', b'S') else 1)
    lon = dms(g[4]) * (-1 if g.get(3) in ('W', b'W') else 1)
    return (round(lat, 6), round(lon, 6)) if lat and lon else None

CODE = {0: 'D00', 1: 'D10', 2: 'D20', 3: 'D40'}
def to_idx(name):
    n = name.lower()
    return 0 if 'long' in n else 1 if 'trans' in n else 2 if 'allig' in n else 3

os.makedirs('annotated', exist_ok=True)
rows, survey, no_gps = [], [], []
for path in imgs:
    gps = get_gps(path)
    if gps is None:
        no_gps.append(os.path.basename(path)); continue
    img = Image.open(path).convert('RGB')
    r = model.predict(img, conf=0.25, imgsz=1024, verbose=False)[0]
    survey.append({'photo': os.path.basename(path), 'latitude': gps[0], 'longitude': gps[1], 'damage_found': len(r.boxes)})
    if len(r.boxes):
        Image.fromarray(r.plot()[:, :, ::-1]).save('annotated/' + os.path.splitext(os.path.basename(path))[0] + '.jpg')
    for k, (c, cf, (x, y, w, h)) in enumerate(zip(r.boxes.cls.tolist(), r.boxes.conf.tolist(), r.boxes.xywhn.tolist())):
        t = to_idx(model.names[int(c)])
        size = round(min(3.0, max(0.3, (w * h) / 0.02)), 2)       # 1.0 = average-sized damage
        # spread several detections in one photo by ~1 m so they do not sit on top of each other
        rows.append({'latitude': gps[0] + k * 0.00001, 'longitude': gps[1], 'type': CODE[t],
                     'damage_type': model.names[int(c)], 'size': size, 'confidence': round(cf, 3),
                     'photo': os.path.basename(path)})

det = pd.DataFrame(rows); pts = pd.DataFrame(survey)
print(f"{len(pts)} photos with GPS, {len(no_gps)} without GPS (skipped)")
print(f"{len(det)} damage detections in {int((pts.damage_found > 0).sum()) if len(pts) else 0} photos")
if len(det): print(det.type.value_counts())
if no_gps: print('No GPS in:', no_gps[:10])

## 4. Check on a map

In [ ]:
import folium
if len(pts):
    m = folium.Map(location=[pts.latitude.mean(), pts.longitude.mean()], zoom_start=15,
                   tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}', attr='Esri')
    for _, p in pts.iterrows():
        folium.CircleMarker([p.latitude, p.longitude], radius=4, color='gray', fill=True, popup=p.photo).add_to(m)
    COL = {'D00': 'blue', 'D10': 'purple', 'D20': 'orange', 'D40': 'red'}
    for _, d in det.iterrows():
        folium.CircleMarker([d.latitude, d.longitude], radius=7, color=COL[d.type], fill=True, fill_opacity=.9,
                            popup=f"{d.damage_type} ({d.confidence}) · {d.photo}").add_to(m)
    display(m)

## 5. Download the dataset

In [ ]:
det.to_csv('chennai_damage_survey.csv', index=False)
pts.to_csv('chennai_survey_photos.csv', index=False)
!zip -qr annotated.zip annotated
files.download('chennai_damage_survey.csv')
files.download('chennai_survey_photos.csv')
files.download('annotated.zip')